In [ ]:
from llama_cpp import Llama
import json



# 載入模型 (已關閉底層煩人的 C++ 日誌)
llm = Llama(
    model_path=r"C:\Users\user\Desktop\llm test\Mistral-Nemo-Instruct-2407-Q4_K_M.gguf",
    n_gpu_layers=20,      # 從 35 降到 20，讓 CPU 多幫忙扛一點，確保不爆顯存
    n_ctx=2048,
    n_batch=256,
    flash_attn=True,      # 🚀 救星參數：開啟 Flash Attention 大幅減少 VRAM 佔用
    verbose=False
)
print("✅ 模型載入完成！開始現場測試！")
print("-" * 50)

while True:
    # 讓你在終端機直接輸入
    target_word = input("\n👉 請輸入你想測試的中文單字 (輸入 'q' 離開程式): ")
    
    if target_word.lower() == 'q':
        print("👋 測試結束，掰掰！")
        break
        
    if not target_word.strip():
        continue

    # 組合 Prompt
    prompt = f"""[INST] 你是一個專業的台灣客語教師。請根據輸入的中文單字，生成一句適合國小生學習的生活化客語例句（長度在 10 個字以內）。
請務必只輸出合法的 JSON 格式，包含 "hakka_sentence" 與 "chinese_translation" 兩個鍵值，絕對不要輸出任何其他說明文字。

輸入：椅子 [/INST]
{{"hakka_sentence": "這張椅子當好坐。", "chinese_translation": "這張椅子很好坐。"}}

[INST] 輸入：吃飯 [/INST]
{{"hakka_sentence": "大家來食飯囉。", "chinese_translation": "大家來吃飯囉。"}}

[INST] 輸入：{target_word} [/INST]
"""
    
    print("🧠 模型思考中...")
    
    try:
        # 呼叫模型推論
        response = llm(
            prompt,
            max_tokens=100,
            temperature=0.2,
            echo=False
        )
        
        # 抓取純文字結果
        result_text = response["choices"][0]["text"].strip()
        
        print("\n" + "="*40)
        print("✨ 【模型原始輸出 (Raw Output)】:")
        print(result_text)
        print("="*40)
        
        # 幫你驗證是不是真的 JSON
        try:
            parsed_json = json.loads(result_text)
            print("🟢 格式檢查：完美！這是一個標準的 JSON 格式，後端保證接得到。")
        except json.JSONDecodeError:
            print("🔴 格式檢查：失敗！模型這次輸出的不是標準 JSON，可能參雜了廢話。")
            
    except Exception as e:
        print(f"❌ 發生錯誤: {e}")

llama_new_context_with_model: n_ctx_per_seq (2048) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


✅ 模型載入完成！開始現場測試！
--------------------------------------------------



👉 請輸入你想測試的中文單字 (輸入 'q' 離開程式):  蘋果


🧠 模型思考中...

✨ 【模型原始輸出 (Raw Output)】:
{"hakka_sentence": "我好喜歡食蘋果。", "chinese_translation": "我好喜歡吃蘋果。"}
🟢 格式檢查：完美！這是一個標準的 JSON 格式，後端保證接得到。



👉 請輸入你想測試的中文單字 (輸入 'q' 離開程式):  桌子


🧠 模型思考中...

✨ 【模型原始輸出 (Raw Output)】:
{"hakka_sentence": "書本放桌子上。", "chinese_translation": "書本放桌子上。"}
🟢 格式檢查：完美！這是一個標準的 JSON 格式，後端保證接得到。


In [10]:
from llama_cpp import Llama
import json
import re

# 載入模型
llm = Llama(
    model_path=r"C:\Users\user\Desktop\llm test\Mistral-Nemo-Instruct-2407-Q4_K_M.gguf",
    n_gpu_layers=20,
    n_ctx=2048,
    n_batch=256,
    flash_attn=True,
    verbose=False
)
print("✅ 模型載入完成！開始連續造句測試！")
print("-" * 50)

while True:
    # 讓你在終端機直接輸入，提示可以使用空格分隔
    user_input = input("\n👉 請輸入 1 到 5 個中文單字，請用空格分隔 (輸入 'q' 離開): ")
    
    if user_input.lower() == 'q':
        print("👋 測試結束，掰掰！")
        break
        
    if not user_input.strip():
        continue

    # 將輸入字串切割成列表，並限制最多取前五個單字
    word_list = [w.strip() for w in user_input.split() if w.strip()][:5]
    words_str = "、".join(word_list)

    # 組合 Prompt，要求關聯性、字數約20字、以及 JSON 陣列格式
    prompt = f"""[INST] 你是一個專業的中文寫作助手。請根據我提供的多個單字，為每個單字造一個中文句子。
要求：
1. 這些句子必須構成一個有關聯的連續情境或故事。
2. 每個句子的長度必須控制在大約 20 個字左右。
3. 請務必只輸出合法的 JSON 陣列 (Array) 格式，包含 "word" 與 "sentence" 兩個鍵值，絕對不要輸出任何其他說明文字。

輸入單字：蘋果、公園、下雨 [/INST]
[
  {{"word": "蘋果", "sentence": "他手裡拿著一顆紅透的蘋果，心情看起來非常愉快。"}},
  {{"word": "公園", "sentence": "我們原本約好要在這座寬敞的公園裡一起野餐吃水果。"}},
  {{"word": "下雨", "sentence": "沒想到天空突然下雨，打亂了所有原本規劃好的行程。"}}
]

[INST] 輸入單字：{words_str} [/INST]
"""
    
    print(f"🧠 模型思考中 (處理單字: {words_str})...")
    
    try:
        response = llm(
            prompt,
            max_tokens=400,
            temperature=0.3, 
            echo=False,
            stop=["[INST]", "```"] 
        )
        
        # 使用 chr(96) 動態生成三個反引號，徹底避開環境解析 Bug
        backticks = chr(96) * 3
        
        # 抓取純文字結果
        result_text = response["choices"][0]["text"].strip()
        
        # 依序拔除可能出現的 json 標記與單純的反引號標記
        cleaned_text = result_text.replace(f"{backticks}json\n", "").replace(f"{backticks}json", "").replace(backticks, "").strip()
        
        print("\n" + "="*40)
        print("✨ 【模型原始輸出 (Raw Output)】:")
        print(result_text)
        print("="*40)
        
        try:
            parsed_json = json.loads(cleaned_text)
            print("🟢 格式檢查：完美！這是一個標準的 JSON 陣列格式。")
            print("📦 解析結果：")
            for item in parsed_json:
                print(f"  [{item.get('word', '未知')}] -> {item.get('sentence', '無句子')}")
        except json.JSONDecodeError:
            print("🔴 格式檢查：失敗！模型這次輸出的不是標準 JSON。")
            
    except Exception as e:
        print(f"❌ 發生錯誤: {e}")

llama_new_context_with_model: n_ctx_per_seq (2048) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


✅ 模型載入完成！開始連續造句測試！
--------------------------------------------------



👉 請輸入 1 到 5 個中文單字，請用空格分隔 (輸入 'q' 離開):  桌子 駱駝 樹 膝蓋 望遠鏡


🧠 模型思考中 (處理單字: 桌子、駱駝、樹、膝蓋、望遠鏡)...

✨ 【模型原始輸出 (Raw Output)】:
[
  {"word": "桌子", "sentence": "在這張舊桌子上，我發現了一本未完成的日記。"},
  {"word": "駱駝", "sentence": "在撒哈拉沙漠的邊緣，我看到一頭駱駝正在吃著稀少的植物。"},
  {"word": "樹", "sentence": "這棵大樹下，有一個小木屋，是我兒時最喜歡的藏身之地。"},
  {"word": "膝蓋", "sentence": "她膝蓋上放著一本厚厚的書，全神貫注地讀著。"},
  {"word": "望遠鏡", "sentence": "我調整著望遠鏡，想要看得更清楚一些。"}
]
🟢 格式檢查：完美！這是一個標準的 JSON 陣列格式。
📦 解析結果：
  [桌子] -> 在這張舊桌子上，我發現了一本未完成的日記。
  [駱駝] -> 在撒哈拉沙漠的邊緣，我看到一頭駱駝正在吃著稀少的植物。
  [樹] -> 這棵大樹下，有一個小木屋，是我兒時最喜歡的藏身之地。
  [膝蓋] -> 她膝蓋上放著一本厚厚的書，全神貫注地讀著。
  [望遠鏡] -> 我調整著望遠鏡，想要看得更清楚一些。



👉 請輸入 1 到 5 個中文單字，請用空格分隔 (輸入 'q' 離開):  獅子 老虎 河流


🧠 模型思考中 (處理單字: 獅子、老虎、河流)...

✨ 【模型原始輸出 (Raw Output)】:
[
  {"word": "獅子", "sentence": "非洲大草原上，一隻威武的獅子正在休息。"},
  {"word": "老虎", "sentence": "在印度的密林中，一隻老虎正在等待著猎物靠近。"},
  {"word": "河流", "sentence": "這條河流將它們分隔開來，各自守護著自己的領地。"}
]
🟢 格式檢查：完美！這是一個標準的 JSON 陣列格式。
📦 解析結果：
  [獅子] -> 非洲大草原上，一隻威武的獅子正在休息。
  [老虎] -> 在印度的密林中，一隻老虎正在等待著猎物靠近。
  [河流] -> 這條河流將它們分隔開來，各自守護著自己的領地。


KeyboardInterrupt: Interrupted by user


👉 請輸入 1 到 5 個中文單字，請用空格分隔 (輸入 'q' 離開):  q
